In [ ]:
# ruff: noqa: T201, S311

import contextlib
import pathlib
import random
import time
import warnings
from collections.abc import Generator

import cache_pandas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tqdm
from typing_extensions import Callable, Final, cast

import svglab


SOURCE_DIR: Final = pathlib.Path("~/Downloads/freesvg").expanduser()
MIN_SIZE: Final = 10 * 1000  # 10 KB
MAX_SIZE: Final = 50 * 1000  # 50 KB
SAMPLES: Final = 1000
SEED: Final = 42
OUTPUT: Final = pathlib.Path("output.csv")

FORMATTER: Final = svglab.Formatter(
    color_mode="original",
    coordinate_precision=svglab.FloatPrecisionSettings(fallback=1),
    indent=2,
    large_number_scientific_threshold=None,
    path_data_coordinates="absolute",
    path_data_shorthand_curve_commands="original",
    path_data_shorthand_line_commands="original",
    small_number_scientific_threshold=None,
    spaces_around_attrs=False,
    spaces_around_function_args=False,
    strip_leading_zero=False,
    xmlns="original",
)


random.seed(SEED)
rng: Final = np.random.default_rng(SEED)


@contextlib.contextmanager
def _timer() -> Generator[Callable[[], float]]:
    start = time.perf_counter()

    def elapsed() -> float:
        end = time.perf_counter()
        return (end - start) * 1000

    yield elapsed


def _should_include(name: str) -> bool:
    with (SOURCE_DIR / name).open("r") as f, _timer() as get_elapsed:
        try:
            svg = svglab.parse_svg(f)

            if svg.width is None or svg.height is None:
                return False

            w, h = float(svg.width), float(svg.height)
            svg.set_viewbox((0, 0, 2 * w, 2 * h))
            svg.reify()
        except Exception:  # noqa: BLE001
            return False

        if (elapsed := get_elapsed()) > 10 * 1000:  # 10 s
            warnings.warn(
                f"SVG {name} took too long to process: {elapsed:.2f} ms",
                stacklevel=2,
            )

    return True


@cache_pandas.cache_to_csv("raw.csv")
def _load_data() -> pd.DataFrame:
    data = pd.DataFrame(
        (
            (path.name, path.stat().st_size)
            for path in SOURCE_DIR.glob("*.svg")
        ),
        columns=["filename", "size"],
    )
    return data.sort_values("filename")


@cache_pandas.cache_to_csv("good_size.csv")
def _filter_size(data: pd.DataFrame) -> pd.DataFrame:
    return data[(data["size"] > MIN_SIZE) & (data["size"] < MAX_SIZE)]


@cache_pandas.cache_to_csv("sampled.csv")
def _sample_data(data: pd.DataFrame) -> pd.DataFrame:
    bins = np.linspace(MIN_SIZE, MAX_SIZE, SAMPLES + 1)
    samples = []

    for i in tqdm.tqdm(range(SAMPLES), desc="Sampling SVGs"):
        lower = bins[i]
        upper = bins[i + 1]

        mask = (data["size"] >= lower) & (data["size"] < upper)
        bin_data = data[mask]

        while True:
            sample = bin_data.sample(n=1, random_state=rng)
            filename = sample["filename"].to_numpy()[0]

            if _should_include(filename):
                samples.append(sample)
                break

    return pd.concat(samples, ignore_index=True)


print("Loading SVGs...")
data = _load_data()
print(f"Loaded {len(data)} SVG files.")

print(f"Filtering SVGs by size ({MIN_SIZE} - {MAX_SIZE} bytes)...")
data = _filter_size(data)
print(f"Filtered down to {len(data)} SVG files.")

data = _sample_data(data)
print(f"Sampled {SAMPLES} SVG files:")
display(data.head())
print(data["size"].describe())
print(f"Median: {data['size'].median()} bytes")

# plot the size distribution using a histogram
plt.figure(figsize=(10, 6))
plt.hist(
    data["size"] / 1000,
    bins=(MAX_SIZE - MIN_SIZE) // 1000,
    color="blue",
    alpha=0.7,
    edgecolor="black",
)
plt.title("SVG File Size Distribution")
plt.xlabel("File Size (kB)")
plt.ylabel("Frequency")
plt.grid(axis="y", alpha=0.75)
plt.show()

print(f"Running benchmarks on {SAMPLES} SVG files...")

for filename, _ in data.itertuples(index=False):
    with (SOURCE_DIR / cast(str, filename)).open("r") as f:
        markup = f.read()

    with _timer() as get_elapsed:
        svg = svglab.parse_svg(markup)
        data.loc[data["filename"] == filename, "parse_time"] = (
            get_elapsed()
        )

    assert svg.width is not None
    assert svg.height is not None
    w, h = float(svg.width), float(svg.height)

    with _timer() as get_elapsed:
        svg.set_viewbox((0, 0, 2 * w, 2 * h))
        svg.reify()
        data.loc[data["filename"] == filename, "reify_time"] = (
            get_elapsed()
        )

    element = random.choice(
        [
            *svg.find_all(svglab.GraphicsElement, recursive=False),
            *svg.find_all(svglab.ContainerElement, recursive=False),
        ]
    )

    with _timer() as get_elapsed:
        element.get_mask(visible_only=True)
        data.loc[data["filename"] == filename, "mask_time"] = get_elapsed()

    with _timer() as get_elapsed:
        svg.to_xml(formatter=FORMATTER)
        data.loc[data["filename"] == filename, "serialize_time"] = (
            get_elapsed()
        )

data.to_csv(OUTPUT, index=False)